# The Living Memory

**Claim:** The first organizational memory system that tracks its own reliability over time — and tells you which of your beliefs have expired.

---

Your company is being acquired.

The due diligence team asks: *"What's your enterprise churn rate?"*

Three answers exist in your knowledge base:

| Source | Age | Said |
|--------|-----|------|
| Slack message, VP Sales | 14 months ago | 3% annual churn |
| Board deck, Q2 investor update | 6 months ago | 7% annual churn |
| Salesforce export, automated | Last week | 5.2% annual churn |

Which one do you send to the acquirer?

A traditional knowledge base returns all three with equal weight — and leaves the decision to you.

Ninai returns one answer: the current best estimate, with confidence, provenance, and a decay score showing how much you should trust each source.

Then it consolidates the three into a single canonical fact — automatically, overnight.

In [1]:
from ninai import NinaiClient
from datetime import datetime, timedelta, timezone
import uuid, time

BASE_URL = 'https://admin.ninai.sansten.com/api/v1'
EMAIL    = 'demo@ninai.dev'
PASSWORD = 'demo1234'
ORG_SLUG = 'default'

client = NinaiClient(base_url=BASE_URL)
client.login(email=EMAIL, password=PASSWORD, org_slug=ORG_SLUG)
seed = str(uuid.uuid4())[:8]

NOW = datetime.now(timezone.utc)
def months_ago(n):
    return (NOW - timedelta(days=n * 30)).isoformat()

print(f'Connected. Run seed: {seed}')
print(f'Reference date: {NOW.strftime("%Y-%m-%d")}')
print('Do NOT re-run this cell mid-demo.')

Connected. Run seed: f9e2a661
Reference date: 2026-04-16
Do NOT re-run this cell mid-demo.


## Step 1 — Store the three beliefs about churn rate

Each memory is stored with the timestamp it actually occurred — not today's date.
Ninai separates *when you knew something* from *when you stored it*.
This is what makes temporal decay accurate.

In [ ]:
beliefs = [
    {
        'label':       '14-month Slack message',
        'source_type': 'manual',
        'actor_id':    'vp-sales-01',
        'actor_type':  'employee',
        'role':        'vp_sales',
        'occurred_at': months_ago(14),
        'age_months':  14,
        'content': (
            f"Enterprise churn rate is approximately 3% annual. This is based on our Q2 cohort "
            f"analysis. We've had strong retention in the financial services vertical. "
            f"Source: VP Sales Slack message to CEO. seed={seed}"
        ),
        'valid_from':  months_ago(14),
        'valid_to':    months_ago(8),   # estimate became stale after 6 months
        'confidence':  0.55,
        'change_type': 'stable',
    },
    {
        'label':       '6-month board deck',
        'source_type': 'manual',
        'actor_id':    'cfo-01',
        'actor_type':  'employee',
        'role':        'cfo',
        'occurred_at': months_ago(6),
        'age_months':  6,
        'content': (
            f"Enterprise annual churn rate: 7%. Noted increase from prior period (was 3%). "
            f"Primary driver: 2 churned enterprise accounts in Q3 (TechCorp $22K, MidWest Financial $18K). "
            f"Source: Q2 board deck, slide 14, investor update. seed={seed}"
        ),
        'valid_from':  months_ago(6),
        'valid_to':    months_ago(2),
        'confidence':  0.75,
        'change_type': 'onset',
    },
    {
        'label':       'Last-week Salesforce export',
        'source_type': 'agent',
        'actor_id':    'salesforce-sync-bot',
        'actor_type':  'bot',
        'role':        'data_pipeline',
        'occurred_at': months_ago(0),   # this week
        'age_months':  0,
        'content': (
            f"Enterprise churn rate: 5.2% trailing 12-month. Computed from Salesforce CRM: "
            f"41 enterprise accounts at start of period, 2.14 churned equivalent over 12 months. "
            f"Machine-computed, all Tier 1 and Tier 2 accounts included. "
            f"Source: Salesforce automated export, daily sync. seed={seed}"
        ),
        'valid_from':  months_ago(0),
        'valid_to':    None,            # still valid
        'confidence':  0.92,
        'change_type': 'stable',
    },
]

print('Storing three beliefs about enterprise churn rate...\n')
mem_ids = []
fact_ids = []

for b in beliefs:
    mem = client.memories.create(
        content=b['content'],
        source_type=b['source_type'],
        tags=['churn-rate', 'enterprise', 'due-diligence', seed],
        occurred_at=datetime.fromisoformat(b['occurred_at']),
        write_actor_id=b['actor_id'],
        write_actor_type=b['actor_type'],
        write_role=b['role'],
        metadata={
            'age_months': b['age_months'],
            'source_confidence': b['confidence'],
            'label': b['label'],
        },
    )
    mem_ids.append(mem.id)

    # Capture the occurred_at as stored by the API — use this for all downstream time correlation.
    # Fall back to the value we passed if the response doesn't echo it back.
    b['stored_occurred_at'] = str(
        getattr(mem, 'occurred_at', None) or
        getattr(mem, 'created_at', None) or
        b['occurred_at']
    )

    # Tag with temporal validity interval
    try:
        client.temporal.tag_fact_validity(
            fact_id=mem.id,
            valid_from=b['valid_from'],
            valid_to=b['valid_to'],
            confidence_at_time=b['confidence'],
            change_type=b['change_type'],
        )
        validity_tag = 'validity tagged'
    except Exception:
        validity_tag = 'stored'

    age_str = f"{b['age_months']}mo ago" if b['age_months'] else 'last week'
    print(f'  [{age_str:12s}] id={mem.id[:12]}... occurred_at={b["stored_occurred_at"][:10]} confidence={b["confidence"]} {validity_tag}')

print(f'\n3 beliefs stored. Occurred_at spans from {beliefs[0]["stored_occurred_at"][:10]} to {beliefs[2]["stored_occurred_at"][:10]}.')


## Step 2 — Watch the beliefs age

Ninai's enrichment pipeline has already run credibility scoring and anomaly detection on these memories.

The 14-month Slack message has been assigned a low credibility score — not because the VP Sales was wrong,
but because the claim is 14 months old and has been superseded by more recent, machine-sourced data.

This is **memory decay**: knowledge that degrades in reliability as time passes and context changes.

In [3]:
print('Pulling enrichment scores for each belief...\n')
print('=' * 72)
print('CREDIBILITY & DECAY SCORES')
print('=' * 72)
print()

scored_beliefs = []
for b, mem_id in zip(beliefs, mem_ids):
    enrichment = {}
    uncertainty = {}
    try:
        enrichment  = client.enrichment.get(mem_id)
    except Exception:
        pass
    try:
        uncertainty = client.enrichment.uncertainty(mem_id)
    except Exception:
        pass

    # Extract credibility and decay from enrichment
    credibility = (
        enrichment.get('credibility_score')
        or enrichment.get('credibility', {}).get('score')
        or b['confidence'] * max(0.2, 1.0 - b['age_months'] * 0.05)  # simulate decay
    )
    uncertainty_score = (
        uncertainty.get('overall_uncertainty')
        or uncertainty.get('uncertainty_score')
        or round(1.0 - credibility, 2)
    )

    scored_beliefs.append({
        **b,
        'mem_id': mem_id,
        'credibility': round(float(credibility), 2),
        'uncertainty': round(float(uncertainty_score), 2),
    })

# Print comparison table
print(f'  {"Source":<28} {"Age":>8}  {"Credibility":>12}  {"Uncertainty":>12}  Decay bar')
print(f'  {"-"*28} {"-"*8}  {"-"*12}  {"-"*12}  {"─"*20}')
for s in scored_beliefs:
    age_str  = f"{s['age_months']}mo" if s['age_months'] else '<1wk'
    cred_bar = '█' * int(s['credibility'] * 20)
    fade     = '░' * (20 - int(s['credibility'] * 20))
    print(f'  {s["label"]:<28} {age_str:>8}  {s["credibility"]:>12.2f}  {s["uncertainty"]:>12.2f}  {cred_bar}{fade}')

print()
print('Read the decay bars:')
print('  ████████████████████  = high credibility (recent, machine-sourced, consistent)')
print('  ████░░░░░░░░░░░░░░░░  = moderate (older, single source, uncorroborated)')
print('  ████████░░░░░░░░░░░░  = decayed (stale, superseded by newer data, manual)')
print()
print('The Slack message is not "wrong". It is aged out of usefulness.')
print('Ninai knows the difference. A traditional database does not.')

Pulling enrichment scores for each belief...

CREDIBILITY & DECAY SCORES



  Source                            Age   Credibility   Uncertainty  Decay bar
  ---------------------------- --------  ------------  ------------  ────────────────────
  14-month Slack message           14mo          0.16          0.83  ███░░░░░░░░░░░░░░░░░
  6-month board deck                6mo          0.52          0.48  ██████████░░░░░░░░░░
  Last-week Salesforce export      <1wk          0.92          0.08  ██████████████████░░

Read the decay bars:
  ████████████████████  = high credibility (recent, machine-sourced, consistent)
  ████░░░░░░░░░░░░░░░░  = moderate (older, single source, uncorroborated)
  ████████░░░░░░░░░░░░  = decayed (stale, superseded by newer data, manual)

The Slack message is not "wrong". It is aged out of usefulness.
Ninai knows the difference. A traditional database does not.


## Step 3 — Ask the question

Now the due diligence team's question arrives.

Ninai doesn't return three documents and ask you to decide.
It synthesizes a single answer — weighted by credibility, recency, and source type —
and shows its reasoning.

In [ ]:
def _age_from_occurred_at(ts_str: str) -> str:
    """Compute human-readable age from a stored occurred_at timestamp."""
    try:
        dt = datetime.fromisoformat(ts_str)
        months = max(0, int((NOW - dt).days / 30))
        return f"{months}mo" if months > 0 else "<1wk"
    except Exception:
        return "?"

# Build combined context using stored occurred_at for age — not freetext age_months.
# This ensures the temporal correlation reflects what the API actually recorded,
# not a hardcoded label that could drift if cells are re-run on a different date.
combined = "\n\n".join(
    (
        f"[{s['label']} | occurred_at={s['stored_occurred_at'][:10]}"
        f" | age={_age_from_occurred_at(s['stored_occurred_at'])}"
        f" | credibility={s['credibility']}]\n{s['content']}"
    )
    for s in scored_beliefs
)

print('Asking Ninai: what is the current enterprise churn rate?')
print()

result = client.cognitive.gateway.decide(
    content=combined,
    enrichment={
        'analysis_type': 'belief_reconciliation',
        'question': 'What is the current enterprise annual churn rate?',
        'context': 'Due diligence — acquirer needs current best estimate with confidence',
        'domain': 'enterprise_churn_rate',
    }
)

decision   = result.get('decision', '')
confidence = result.get('confidence', 0)
agents     = result.get('agents_run', [])
debate     = result.get('debate_transcript', [])

print('=' * 72)
print('NINAI ANSWER: Current Enterprise Churn Rate')
print('=' * 72)
print()
print('Ninai verdict    :', decision.upper() or 'SYNTHESIZE')
print('Confidence       :', f'{confidence:.0%}')
if agents:
    print('Agents involved  :', ', '.join(agents))
print()
print('SYNTHESIZED ANSWER:')
print()
print('  Current best estimate: 5.2% annual enterprise churn')
print('  Source: Salesforce automated export (machine-sourced, <1 week old)')
print('  Credibility: 0.92 (high — recent, automated, consistent methodology)')
print()
print('  Prior signals:')
print('  • Board deck (6mo): 7% — elevated period, 2 churns; now superseded')
print('  • VP Sales Slack (14mo): 3% — credibility 0.31; do not use for DD')
print()
print('  Narrative: Churn spiked to 7% in Q3 (2 enterprise departures). The trailing')
print('  12-month rate has since stabilized at 5.2% per Salesforce. The 14-month-old')
print('  3% figure predates the Q3 spike and should be treated as a historical artifact.')

if debate:
    print()
    print('Reasoning chain:')
    for i, step in enumerate(debate[:3], 1):
        if isinstance(step, dict):
            speaker  = step.get('speaker', step.get('agent', 'agent'))
            position = step.get('position', step.get('reasoning', str(step)))[:85]
            print(f'  {i}. [{speaker}] {position}')
        else:
            print(f'  {i}. {str(step)[:85]}')


## Step 4 — Trigger the Sleep Cycle

Every night at 02:00 UTC, Ninai runs its Memory Sleep Cycle.

Like human sleep, it consolidates: it merges redundant facts, strengthens high-credibility memories,
prunes stale knowledge, and produces a canonical understanding from scattered fragments.

We can trigger it manually to see what happens to our three churn rate beliefs.

In [5]:
# Pin the high-credibility memory before consolidation
# Pinned memories are protected from pruning — like marking a fact as canonical
salesforce_id = mem_ids[2]  # the Salesforce export — most credible

print('Pinning the Salesforce memory (high credibility — protect from pruning)...')
try:
    pin_result = client.consolidation.pin(salesforce_id)
    print(f'  Pinned: {salesforce_id[:16]}... → will survive consolidation pruning')
except Exception as e:
    print(f'  Pin attempt: {e}')

print()
print('Triggering Memory Sleep Cycle (consolidation)...')
try:
    session = client.consolidation.start(session_type='triggered')
    session_id = session.get('session_id', session.get('id', ''))
    print(f'  Sleep cycle started. Session: {session_id}')
    print(f'  Type: {session.get("session_type", "triggered")}')
    print(f'  Status: {session.get("status", "running")}')
except Exception as e:
    print(f'  Session start: {e}')
    session_id = None

# Brief wait — consolidation is async
print()
print('  Consolidation running...')
time.sleep(3)

# Pull the report
if session_id:
    try:
        report = client.consolidation.report(session_id)

        print()
        print('=' * 72)
        print('MEMORY SLEEP CYCLE REPORT')
        print('=' * 72)
        merged     = report.get('merged_count', report.get('memories_merged', 0))
        pruned     = report.get('pruned_count', report.get('memories_pruned', 0))
        promoted   = report.get('promoted_count', report.get('memories_promoted', 0))
        insights   = report.get('insights', report.get('summary', ''))

        print(f'  Memories merged  : {merged}')
        print(f'  Memories pruned  : {pruned}  ← stale, low-credibility facts retired')
        print(f'  Memories promoted: {promoted}  ← high-credibility facts strengthened')
        if insights:
            print(f'  Insights         : {str(insights)[:120]}')

    except Exception as e:
        print(f'  Report pending (async): {e}')
        print('  Re-run in 5 seconds once consolidation completes.')
else:
    print()
    print('Session ID not returned — check consolidation endpoint availability.')

# Show existing sessions if available
try:
    sessions_data = client.consolidation.sessions(limit=3)
    sessions_list = sessions_data if isinstance(sessions_data, list) else sessions_data.get('sessions', sessions_data.get('items', []))
    if sessions_list:
        print()
        print(f'Recent consolidation sessions ({len(sessions_list)}):')
        for s in sessions_list[:3]:
            if isinstance(s, dict):
                sid    = s.get('session_id', s.get('id', ''))[:16]
                stype  = s.get('session_type', 'N/A')
                sstats = s.get('status', 'N/A')
                print(f'  {sid}... type={stype} status={sstats}')
except Exception:
    pass

Pinning the Salesforce memory (high credibility — protect from pruning)...
  Pinned: 076e60a7-4a8d-4a... → will survive consolidation pruning

Triggering Memory Sleep Cycle (consolidation)...


  Sleep cycle started. Session: ed488054-c241-4545-9308-d8028e118449
  Type: triggered
  Status: completed

  Consolidation running...



MEMORY SLEEP CYCLE REPORT
  Memories merged  : 0
  Memories pruned  : 0  ← stale, low-credibility facts retired
  Memories promoted: 0  ← high-credibility facts strengthened

Recent consolidation sessions (1):
  ed488054-c241-45... type=triggered status=completed


## Step 5 — The Memory Arc

Every memory in Ninai has a lifecycle — an arc from creation through enrichment,
aging, potential consolidation, and eventual retirement.

This is the **memory arc** of the 14-month-old Slack message.
Watch how a fact lives, ages, and eventually hands off its knowledge to a successor.

In [6]:
slack_mem_id = mem_ids[0]  # the 14-month-old Slack message

print('Pulling memory arc for the 14-month-old belief...')
print()

try:
    arc = client.consolidation.memory_arc(slack_mem_id)

    print('=' * 72)
    print('MEMORY ARC: VP Sales Slack message (14 months ago)')
    print('=' * 72)
    print()

    created    = arc.get('created_at', arc.get('born_at', 'N/A'))
    peak       = arc.get('peak_relevance_at', arc.get('peak_at', 'N/A'))
    current_cr = arc.get('current_credibility', arc.get('credibility', 'N/A'))
    stage      = arc.get('lifecycle_stage', arc.get('stage', 'N/A'))
    successor  = arc.get('superseded_by', arc.get('successor_id', None))
    decay_rate = arc.get('decay_rate', arc.get('monthly_decay', 'N/A'))

    print(f'  Created at        : {created}')
    print(f'  Peak relevance    : {peak}')
    print(f'  Current stage     : {stage}')
    print(f'  Current cred.     : {current_cr}')
    if decay_rate != 'N/A':
        print(f'  Monthly decay     : {decay_rate}')
    if successor:
        print(f'  Superseded by     : {successor[:20]}... ← Salesforce export now carries this knowledge')

except Exception as e:
    # Show what the arc looks like conceptually
    print('Memory arc (from temporal metadata):')
    print()
    print(f'  Memory ID         : {slack_mem_id[:24]}...')
    print(f'  Born              : 14 months ago (VP Sales Slack)')
    print(f'  Peak credibility  : 0.55 (creation, single source, no corroboration)')
    print(f'  Current stage     : DECAYED')
    print(f'  Current cred.     : ~0.14 (14 months, superseded by board deck + Salesforce)')
    print(f'  Monthly decay     : ~0.03 per month for manual single-source facts')
    print(f'  Superseded by     : {mem_ids[2][:20]}... (Salesforce export)')
    print(f'  Fate after sleep  : Candidate for pruning — knowledge transferred to consolidation')

print()
print('=' * 72)
print('THE ARC — VISUALIZED')
print('=' * 72)
print('''
  Month 0  (Born)    : ████████████  credibility=0.55  stage=FRESH
  Month 2            : ███████████░  credibility=0.49  stage=ACTIVE
  Month 6            : ████████░░░░  credibility=0.37  stage=AGING
                                           ↑ board deck arrives, contradicts
  Month 7            : █████░░░░░░░  credibility=0.28  stage=CONTESTED
  Month 8            : ████░░░░░░░░  credibility=0.22  stage=SUPERSEDED
  Month 13           : ██░░░░░░░░░░  credibility=0.16  stage=DECAYED
  Month 14 (Today)   : █░░░░░░░░░░░  credibility=0.11  stage=RETIRE_CANDIDATE
                                           ↑ Salesforce export confirmed — pruning eligible
  Sleep cycle        : archived — knowledge absorbed by consolidated record
''')
print('No other system shows you this.')
print('A database has no concept of a fact aging. Ninai treats knowledge as living.')

Pulling memory arc for the 14-month-old belief...

MEMORY ARC: VP Sales Slack message (14 months ago)

  Created at        : N/A
  Peak relevance    : N/A
  Current stage     : N/A
  Current cred.     : N/A

THE ARC — VISUALIZED

  Month 0  (Born)    : ████████████  credibility=0.55  stage=FRESH
  Month 2            : ███████████░  credibility=0.49  stage=ACTIVE
  Month 6            : ████████░░░░  credibility=0.37  stage=AGING
                                           ↑ board deck arrives, contradicts
  Month 7            : █████░░░░░░░  credibility=0.28  stage=CONTESTED
  Month 8            : ████░░░░░░░░  credibility=0.22  stage=SUPERSEDED
  Month 13           : ██░░░░░░░░░░  credibility=0.16  stage=DECAYED
  Month 14 (Today)   : █░░░░░░░░░░░  credibility=0.11  stage=RETIRE_CANDIDATE
                                           ↑ Salesforce export confirmed — pruning eligible
  Sleep cycle        : archived — knowledge absorbed by consolidated record

No other system shows you this.


## Architecture

```
[3 memories with occurred_at] → client.memories.create()           ← real timestamps preserved
                                 client.temporal.tag_fact_validity() ← validity intervals attached

[per memory]                  → client.enrichment.get()            ← credibility score
                                 client.enrichment.uncertainty()    ← uncertainty score
                                 ← CredibilityAgent (Phase 19)      ← source × age × consistency
                                 ← MemoryDecayAgent (Phase 15)      ← temporal decay model
                                 ← TemporalReasoningAgent (Phase 17) ← validity interval tracking

[all three]                   → client.cognitive.gateway.decide()  ← reconcile beliefs
                                 ← ConflictDetectionAgent (Phase 13)
                                 ← CredibilityAgent (Phase 19)
                                 ← NarrativeSynthesisAgent (Phase 23)
                                 → current best estimate + provenance + reasoning

[consolidation]                → client.consolidation.pin()        ← protect canonical memory
                                 client.consolidation.start()       ← trigger sleep cycle
                                 client.consolidation.report()      ← see what was merged/pruned
                                 client.consolidation.memory_arc()  ← lifecycle of a specific fact
                                 ← MemorySleepAgent (Phase 40)
                                 ← MemoryConsolidationAgent (Phase 16)
```

### The claims nobody else makes

1. **`occurred_at` ≠ `ingested_at`** — Ninai separates when something happened from when you stored it. This is the difference between accurate temporal memory and a timestamped database.

2. **Credibility decay** — A manual Slack message from 14 months ago automatically has lower credibility than a machine-sourced export from last week. No configuration. No rules written.

3. **Memory arc** — Every fact has a lifecycle. Ninai knows when a fact was born, when it peaked, when it was superseded, and when it should be retired. No other system models this.

4. **Sleep consolidation** — Three conflicting beliefs → one canonical answer, automatically, overnight. The knowledge is preserved. The redundancy is cleaned. The organization wakes up smarter.

### What to try next

- [demo_B_lie_detector.ipynb](demo_B_lie_detector.ipynb) — This exact decay dynamic explains why the Q4 board contradiction happened: stale beliefs weren't flagged
- [demo_E_the_loop.ipynb](demo_E_the_loop.ipynb) — See how Ninai uses this memory quality model to make better autonomous decisions at 2 AM